In [ ]:
from pyspark.sql.functions import *

In [ ]:
%py
df_orders_silver = spark.sql("""
    WITH CTE_total_sales AS (
        SELECT
        order_id
        ,quantity
        ,discount
        ,ROUND((list_price*quantity),2) AS total_amount
        ,ROUND((list_price*quantity)-((list_price*quantity)*(discount)),2) AS total_amount_discount
        FROM lugus_databricks_dev.bronze.order_items
    )
    SELECT
    a.order_id
    ,a.customer_id
    ,case 
        when a.order_status = 1 then 'Pending'
        when a.order_status = 2 then 'Processing'
        when a.order_status = 3 then 'Shipped'
        when a.order_status = 4 then 'Delivered'
    end as order_status
    ,d.quantity
    ,a.order_date
    ,b.store_name
    ,b.state
    ,b.city
    ,a.required_date
    ,a.shipped_date
    ,concat(c.last_name,', ' , c.first_name) AS fullname
    ,c.active
    ,c.email
    ,d.total_amount
    ,d.total_amount_discount
    ,d.discount
    FROM lugus_databricks_dev.bronze.orders a
    LEFT JOIN lugus_databricks_dev.bronze.stores b
    ON a.store_id = b.store_id
    LEFT JOIN lugus_databricks_dev.bronze.staffs c
    ON a.staff_id = c.staff_id
    LEFT JOIN CTE_total_sales d
    ON a.order_id = d.order_id
""")

In [ ]:
%py
df_orders_silver.write.mode("overwrite").format("delta").options(
    overwriteSchema = "true"
).saveAsTable('lugus_databricks_dev.silver.orders')